In [ ]:
import sklearn
from sklearn.datasets import make_circles
import torch 
import pandas as pd
import matplotlib.pyplot as plt
from torch import nn

In [ ]:
n_samples=1000
X,y=make_circles(n_samples,noise=0.05,random_state=42)
len(X),len(y)

In [ ]:
circles=pd.DataFrame({"X1":X[:,0],
                      "X2":X[:,1],
                      "label":y})
circles.head(10)

In [ ]:
plt.scatter(x=X[:,0],y=X[:,1],c=y,cmap=plt.cm.RdYlBu)

In [ ]:
X[:5],y[:5]

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,
                                              y,test_size=0.2,random_state=42)
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_test  = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

In [ ]:
len(X_test)

In [ ]:
class CircleModel(nn.Module):
    def __init__(self):
        super().__init__()
        # self.layer_1=nn.Linear(in_features=2,out_features=5)
        # self.layer_2=nn.Linear(in_features=5,out_features=1)
        self.two_linear_layer=nn.Sequential(  # direct sequential implementation
            nn.Linear(in_features=2,out_features=12),
            nn.ReLU(),       
            nn.Linear(in_features=12,out_features=1)
        )
    def forward(self,x):
        # return self.layer_2(self.layer_1(x))
        return self.two_linear_layer(x)
model=CircleModel()
model

In [ ]:
# model=nn.Sequential(
#     nn.Linear(in_features=2,out_features=5),
#     nn.Linear(in_features=5,out_features=1)
# )
# model

In [ ]:
model.state_dict()

In [ ]:
#calculating accuracy
def accuracy_fn(y_true,y_pred):
    correct=torch.eq(y_true,y_pred).sum().item()
    acc=(correct/len(y_pred))*100
    return acc

In [ ]:
model.eval()
with torch.inference_mode():
    y_logits=model(X_test)[:5]
y_logits #now the value we got here are raw values after values are passed thru forward pass

In [ ]:
y_probs=torch.sigmoid(y_logits)
y_probs # as the above values are raw and we cant predict or do anything we need activation function so values pas thru them 
#and we can check if its class 1 or not by rounding of basically sigmoid convert it between 0 and 1

In [ ]:
y_preds=torch.round(y_probs)
y_preds # so now in this cell if values of probs>=0.5 it will be classified as 1 else 0 

# so basically idea is pass thru forward pass get raw input values converted then pass thru acitvation function then round off
# and get final output

In [ ]:
loss_fn=torch.nn.BCEWithLogitsLoss() # combine sigmoid layer and BCE layer together
optimizer=torch.optim.Adam(params=model.parameters(),lr=0.01)


In [ ]:
# the current code is for direct measuring and calculating losses while manual passing of sigmoid and rounding off is basically used for
#iterating and doing same stuff manually and commented one uses BCEWtih logits which has built in sigmoid
epochs=200
train_loss_values = []
test_loss_values = []
epoch_count = []
for epoch in range(epochs):
    model.train()
    y_logits=model(X_train)
    # y_pred_train=model(X_train)
    y_pred_train=torch.round(torch.sigmoid(y_logits))
    # loss=loss_fn(y_pred_train,y_train)
    loss=loss_fn(y_logits,y_train) # here we are using logits instead of y_pred_train becoz logits is the raw value we got from forward pass
    # while if we pass y_pred_train it has round off values 0&1 which will not allow us to manupilate the gradientsn properly nor we can 
    #loss of pred and actual values
    acc=accuracy_fn(y_true=y_train,y_pred=y_pred_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    model.eval()
    with torch.inference_mode():
        y_test_pred=model(X_test)
        y_test_loss=loss_fn(y_test_pred,y_test)
    
    if epoch%10==0:
        epoch_count.append(epoch)
        train_loss_values.append(loss.detach().numpy())
        test_loss_values.append(y_test_loss.detach().numpy())
        print(model.state_dict())
        print(f"Train loss {loss}")
        print(f"Test Loss {y_test_loss}")
        print(epoch)


In [ ]:
import matplotlib.pyplot as plt

plt.plot(epoch_count, train_loss_values, label="Train Loss")
plt.plot(epoch_count, test_loss_values, label="Test Loss")
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# create a grid
x_min, x_max = X[:,0].min()-0.5, X[:,0].max()+0.5
y_min, y_max = X[:,1].min()-0.5, X[:,1].max()+0.5

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 200),
    np.linspace(y_min, y_max, 200)
)

# convert grid to tensor
grid = np.c_[xx.ravel(), yy.ravel()]
grid_tensor = torch.tensor(grid, dtype=torch.float32)

# model predictions
model.eval()
with torch.inference_mode():
    logits = model(grid_tensor)
    probs = torch.sigmoid(logits)
    preds = torch.round(probs)

Z = preds.reshape(xx.shape)

# plot decision boundary
plt.contourf(xx, yy, Z, alpha=0.5)
plt.scatter(X[:,0], X[:,1], c=y, cmap="coolwarm")
plt.title("Decision Boundary")
plt.show()